# Qwen2-VL 증류 학생 — Colab GPU 서버

노트북 PC의 Streamlit 앱(MobileViT 판정·히트맵)은 그대로 두고, **해설을 쓰는 Qwen2-VL만 Colab GPU에서** 돌린다.

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 고른다
2. 위에서부터 셀을 차례로 실행한다 (**런타임 → 모두 실행** 도 된다)
3. 마지막에 나오는 `https://....trycloudflare.com` 주소를 앱 왼쪽 사이드바의 **VLM 서버 주소**에 붙여 넣는다

무료 Colab은 한동안 아무것도 안 하면 끊긴다. 발표 직전에 켜고, 6번 셀로 한 번 워밍업해 둘 것.

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 설치

In [ ]:
!pip install -q -U "transformers>=4.49" peft accelerate
# Colab에 깔린 구버전 torchao 때문에 PEFT가 어댑터를 붙이다 죽는다. 서버는 torchao를 쓰지 않는다.
!pip uninstall -y -q torchao
# 저장소가 비공개라 clone 하지 않는다. 서버에 필요한 파일 3개는 아래 셀에 들어 있다.
!mkdir -p /content/repo/distill
%cd /content/repo

## 3. 서버 코드
저장소의 `distill/` 에서 서버가 쓰는 파일 3개를 그대로 옮겨 적는다.

In [ ]:
%%writefile /content/repo/distill/export_dataset.py
"""3단계: 파이프라인 결과(meta) + 교사 설명(teacher)을 합쳐 QLoRA용 데이터셋으로 내보낸다.

사용법:
    venv\\Scripts\\python.exe distill\\export_dataset.py

결과 (distill/dataset/):
    images/{id}_orig.jpg, _cam.jpg, _crop.jpg
    train.jsonl / val.jsonl / test.jsonl
    stats.json   교사 정확도, 분류기 정확도, 버린 샘플 수
그리고 distill/tiny_genimage_distill.zip (Colab 업로드용)

train/val은 교사 판정이 정답과 맞은 샘플만 남긴다 (틀린 설명을 외우지 않게).
test는 전부 남긴다 (학생 모델을 정답 라벨로 채점하려고).
"""
import argparse
import glob
import json
import os
import random
import shutil
from collections import Counter, defaultdict

VERDICT_KO = {'REAL': '실제 사진', 'AI': 'AI 생성'}
CONF_KO = {'high': '높음', 'medium': '보통', 'low': '낮음'}
USEFUL_KO = {'yes': '판정 근거와 관련 있음', 'partial': '부분적으로 관련 있음', 'no': '판정 근거와 관련 없음'}
REQUIRED = ['id', 'teacher_verdict', 'teacher_confidence', 'observations',
            'heatmap_focus', 'heatmap_useful', 'agrees_with_classifier', 'explanation']


def user_text(m):
    return (
        '첫 번째 이미지는 원본, 두 번째는 분류기의 Grad-CAM 히트맵, '
        '세 번째는 히트맵이 가장 강한 영역을 확대한 크롭입니다.\n'
        f"분류기(MobileViT v2) 판정: {VERDICT_KO[m['mobilevit_pred']]} "
        f"(실제 사진일 확률 {m['mobilevit_prob_real']:.2f})\n"
        '이 이미지가 AI로 생성됐는지 판단하고 근거를 설명하세요.'
    )


def assistant_text(t):
    lines = [f"판정: {VERDICT_KO[t['teacher_verdict']]} (확신: {CONF_KO[t['teacher_confidence']]})", '근거:']
    lines += [f'- {o}' for o in t['observations']]
    lines.append(f"히트맵: {t['heatmap_focus']} ({USEFUL_KO[t['heatmap_useful']]})")
    lines.append('분류기 판정과 일치' if t['agrees_with_classifier'] else '분류기 판정과 불일치')
    lines.append(f"설명: {t['explanation']}")
    return '\n'.join(lines)


DETAIL_KEYS = ['scene', 'evidence_ai', 'evidence_real', 'heatmap_focus_detailed',
               'heatmap_useful_detailed', 'heatmap_comment', 'reasoning',
               'uncertainty', 'explanation_detailed']


def assistant_text_detailed(t):
    # 장면 → 양쪽 근거 → 히트맵 검증 → 저울질 → 결론. 학생이 중간 과정까지 따라 배우게 한다.
    lines = [f"장면: {t['scene']}", '',
             f"판정: {VERDICT_KO[t['teacher_verdict']]} (확신: {CONF_KO[t['teacher_confidence']]})", '']
    lines.append('AI 생성 쪽 근거:')
    lines += [f'- {o}' for o in t['evidence_ai']] or ['- 뚜렷한 것 없음']
    lines.append('실제 촬영 쪽 근거:')
    lines += [f'- {o}' for o in t['evidence_real']] or ['- 뚜렷한 것 없음']
    lines.append('')
    lines.append(f"히트맵: {t['heatmap_focus_detailed']}")
    lines.append(f"히트맵 평가: {USEFUL_KO[t['heatmap_useful_detailed']]} — {t['heatmap_comment']}")
    lines.append('분류기 판정과 일치' if t['agrees_with_classifier'] else '분류기 판정과 불일치')
    lines.append('')
    lines.append(f"판단 과정: {t['reasoning']}")
    if t.get('uncertainty'):
        lines.append(f"불확실한 점: {t['uncertainty']}")
    lines.append('')
    lines.append(f"설명: {t['explanation_detailed']}")
    return '\n'.join(lines)


def valid_detailed(t):
    return (all(k in t for k in DETAIL_KEYS)
            and t['heatmap_useful_detailed'] in USEFUL_KO
            and isinstance(t['evidence_ai'], list) and isinstance(t['evidence_real'], list)
            and isinstance(t['explanation_detailed'], str) and len(t['explanation_detailed']) >= 80)


def valid(t):
    return (all(k in t for k in REQUIRED)
            and t['teacher_verdict'] in VERDICT_KO
            and t['teacher_confidence'] in CONF_KO
            and t['heatmap_useful'] in USEFUL_KO
            and isinstance(t['observations'], list) and 1 <= len(t['observations']) <= 6
            and isinstance(t['explanation'], str) and len(t['explanation']) >= 20)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--work', default='distill/work')
    ap.add_argument('--out', default='distill/dataset')
    ap.add_argument('--n-test', type=int, default=100)
    ap.add_argument('--n-val', type=int, default=50)
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--detailed', action='store_true',
                    help='teacher_full/의 상세 설명을 assistant_text로 쓴다')
    args = ap.parse_args()
    teacher_dir = 'teacher_full' if args.detailed else 'teacher'
    check = (lambda t: valid(t) and valid_detailed(t)) if args.detailed else valid
    render = assistant_text_detailed if args.detailed else assistant_text

    meta = {}
    with open(f'{args.work}/meta.jsonl', encoding='utf-8') as f:
        for line in f:
            m = json.loads(line)
            meta[m['id']] = m

    teacher, bad = {}, []
    for path in sorted(glob.glob(f'{args.work}/{teacher_dir}/batch_*.jsonl')):
        with open(path, encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    t = json.loads(line)
                except json.JSONDecodeError:
                    bad.append(line[:60])
                    continue
                if check(t) and t['id'] in meta:
                    t['agrees_with_classifier'] = t['teacher_verdict'] == meta[t['id']]['mobilevit_pred']
                    teacher[t['id']] = t
                else:
                    bad.append(t.get('id', line[:60]))

    missing = sorted(set(meta) - set(teacher))
    ids = sorted(teacher)

    # 생성기별로 test/val을 고르게 떼어낸다
    rng = random.Random(args.seed)
    by_gen = defaultdict(list)
    for i in ids:
        by_gen[meta[i]['generator']].append(i)
    split = {}
    for need, name in [(args.n_test, 'test'), (args.n_val, 'val')]:
        pool = [i for i in ids if i not in split]
        # 진짜 절반, 가짜는 생성기별 균등
        gens = sorted({meta[i]['generator'] for i in pool})
        fake_gens = [g for g in gens if g != 'Real']
        quota = {'Real': need // 2}
        per, extra = divmod(need - need // 2, len(fake_gens))
        for k, g in enumerate(fake_gens):
            quota[g] = per + (1 if k < extra else 0)
        for g, q in quota.items():
            cands = [i for i in by_gen[g] if i not in split]
            for i in rng.sample(cands, min(q, len(cands))):
                split[i] = name
    for i in ids:
        split.setdefault(i, 'train')

    if os.path.exists(args.out):
        shutil.rmtree(args.out)
    os.makedirs(f'{args.out}/images')

    rows = defaultdict(list)
    dropped_wrong = Counter()
    for i in ids:
        m, t, s = meta[i], teacher[i], split[i]
        correct = t['teacher_verdict'] == m['label']
        if s != 'test' and not correct:
            dropped_wrong[s] += 1
            continue
        imgs = []
        for src, suf in [('std', 'orig'), ('cam', 'cam'), ('crop', 'crop')]:
            shutil.copy(f'{args.work}/{src}/{i}.jpg', f'{args.out}/images/{i}_{suf}.jpg')
            imgs.append(f'images/{i}_{suf}.jpg')
        u, a = user_text(m), render(t)
        rows[s].append({
            'id': i, 'images': imgs,
            'messages': [{'role': 'user', 'content': '<image><image><image>' + u},
                         {'role': 'assistant', 'content': a}],
            'user_text': u, 'assistant_text': a,
            'label': m['label'], 'generator': m['generator'],
            'mobilevit_prob_real': m['mobilevit_prob_real'], 'mobilevit_pred': m['mobilevit_pred'],
            'teacher_verdict': t['teacher_verdict'], 'teacher_correct': correct,
            'heatmap_useful': t['heatmap_useful'],
            **({'heatmap_useful_detailed': t['heatmap_useful_detailed']}
               if 'heatmap_useful_detailed' in t else {}),
        })

    for s in ['train', 'val', 'test']:
        with open(f'{args.out}/{s}.jsonl', 'w', encoding='utf-8') as f:
            for r in rows[s]:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')

    def acc(key):
        per = defaultdict(lambda: [0, 0])
        for i in ids:
            g = meta[i]['generator']
            pred = teacher[i]['teacher_verdict'] if key == 'teacher' else meta[i]['mobilevit_pred']
            per[g][0] += pred == meta[i]['label']
            per[g][1] += 1
        total = sum(c for c, _ in per.values()) / max(1, len(ids))
        return {'overall': round(total, 4), **{g: round(c / n, 3) for g, (c, n) in sorted(per.items())}}

    stats = {
        'samples_built': len(meta), 'teacher_valid': len(ids),
        'teacher_invalid': len(bad), 'teacher_missing': len(missing),
        'teacher_accuracy': acc('teacher'), 'mobilevit_accuracy': acc('mobilevit'),
        'heatmap_useful': dict(Counter(teacher[i]['heatmap_useful'] for i in ids)),
        **({'heatmap_useful_detailed':
            dict(Counter(teacher[i]['heatmap_useful_detailed'] for i in ids))}
           if args.detailed else {}),
        'dropped_teacher_wrong': dict(dropped_wrong),
        'split_sizes': {s: len(rows[s]) for s in ['train', 'val', 'test']},
    }
    with open(f'{args.out}/stats.json', 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    if missing:
        print('설명 없는 id (앞 20개):', missing[:20])

    zip_base = os.path.join(os.path.dirname(args.out), 'tiny_genimage_distill')
    shutil.make_archive(zip_base, 'zip', os.path.dirname(args.out), os.path.basename(args.out))
    print('zip:', zip_base + '.zip')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/repo/distill/train_qlora.py
"""4단계: Qwen2-VL-2B-Instruct에 QLoRA로 교사 설명을 증류한다. (Colab T4 이상에서 실행)

Colab 셀 예시:
    !pip install -q "transformers>=4.49" peft bitsandbytes accelerate pillow
    !unzip -q /content/drive/MyDrive/tiny_genimage_distill.zip -d /content
    !python train_qlora.py --data /content/dataset --out /content/drive/MyDrive/qwen2vl-distill
    !python train_qlora.py --data /content/dataset --out /content/drive/MyDrive/qwen2vl-distill --eval-only

학습: train.jsonl, 검증 손실: val.jsonl
평가(--eval-only): test.jsonl에 대해 설명을 생성하고 "판정:" 줄을 정답 라벨과 비교한다.
"""
import argparse
import json
import os
import re

import torch
from PIL import Image
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoProcessor, BitsAndBytesConfig, Qwen2VLForConditionalGeneration,
                          Trainer, TrainingArguments)

MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'
# 384px 이미지가 장당 약 196토큰이 되도록 고정한다 (--side 로 낮추면 토큰·메모리가 준다)
SIDE = 392
# 비전 인코더는 건드리지 않고 언어 모델 쪽만 LoRA를 붙인다
TARGETS = r'^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$'


def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]


def build_messages(ex, with_answer=True):
    content = [{'type': 'image'}] * len(ex['images']) + [{'type': 'text', 'text': ex['user_text']}]
    msgs = [{'role': 'user', 'content': content}]
    if with_answer:
        msgs.append({'role': 'assistant', 'content': [{'type': 'text', 'text': ex['assistant_text']}]})
    return msgs


def load_images(ex, root):
    return [Image.open(os.path.join(root, p)).convert('RGB') for p in ex['images']]


class Collator:
    def __init__(self, processor, root):
        self.p, self.root = processor, root
        self.head = processor.tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)

    def __call__(self, batch):
        texts = [self.p.apply_chat_template(build_messages(ex), tokenize=False) for ex in batch]
        images = [load_images(ex, self.root) for ex in batch]
        enc = self.p(text=texts, images=images, return_tensors='pt', padding=True)
        labels = enc['input_ids'].clone()
        labels[enc['attention_mask'] == 0] = -100
        # 답변(assistant) 부분만 손실에 넣는다
        n = len(self.head)
        for row in range(labels.size(0)):
            ids = enc['input_ids'][row].tolist()
            start = max(i for i in range(len(ids) - n + 1) if ids[i:i + n] == self.head) + n
            labels[row, :start] = -100
        enc['labels'] = labels
        return enc


def load_base(args):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=args.dtype)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        args.model, quantization_config=bnb, torch_dtype=args.dtype, device_map='auto')
    px = args.side * args.side
    processor = AutoProcessor.from_pretrained(args.model, min_pixels=px, max_pixels=px)
    processor.tokenizer.padding_side = 'right'
    return model, processor


def train(args):
    train_rows = read_jsonl(f'{args.data}/train.jsonl')
    # transformers 5.x는 warmup_ratio를 없애고 warmup_steps만 받는다
    total_steps = max(1, int(len(train_rows) * args.epochs / 8))
    warmup = max(1, round(total_steps * 0.05))
    print(f'총 {total_steps}스텝 / 워밍업 {warmup}스텝', flush=True)
    model, processor = load_base(args)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=args.rank, lora_alpha=args.rank * 2, lora_dropout=0.05,
        target_modules=TARGETS, task_type='CAUSAL_LM'))
    model.print_trainable_parameters()

    targs = TrainingArguments(
        output_dir=args.out, num_train_epochs=args.epochs, learning_rate=args.lr,
        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
        lr_scheduler_type='cosine', warmup_steps=warmup, logging_steps=10,
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model='eval_loss',
        bf16=args.dtype == torch.bfloat16, fp16=args.dtype == torch.float16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
        remove_unused_columns=False, dataloader_num_workers=args.workers, report_to='none')
    trainer = Trainer(model=model, args=targs, data_collator=Collator(processor, args.data),
                      train_dataset=train_rows, eval_dataset=read_jsonl(f'{args.data}/val.jsonl'))
    trainer.train(resume_from_checkpoint=args.resume)
    trainer.save_model(f'{args.out}/final')
    processor.save_pretrained(f'{args.out}/final')


@torch.no_grad()
def evaluate(args):
    model, processor = load_base(args)
    model = PeftModel.from_pretrained(model, f'{args.out}/final').eval()
    processor.tokenizer.padding_side = 'left'
    test = read_jsonl(f'{args.data}/test.jsonl')
    results, correct, parsed = [], 0, 0
    for ex in test:
        prompt = processor.apply_chat_template(build_messages(ex, False), tokenize=False, add_generation_prompt=True)
        enc = processor(text=[prompt], images=[load_images(ex, args.data)], return_tensors='pt').to(model.device)
        out = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False)
        text = processor.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        m = re.search(r'판정:\s*(AI 생성|실제 사진)', text)
        pred = None if not m else ('AI' if m.group(1) == 'AI 생성' else 'REAL')
        parsed += pred is not None
        correct += pred == ex['label']
        results.append({'id': ex['id'], 'label': ex['label'], 'generator': ex['generator'],
                        'student_pred': pred, 'mobilevit_pred': ex['mobilevit_pred'],
                        'teacher_verdict': ex['teacher_verdict'], 'student_text': text})
        print(ex['id'], ex['label'], pred, flush=True)
    n = len(test)
    summary = {
        'n': n, 'format_ok': parsed / n,
        'student_acc': correct / n,
        'teacher_acc': sum(r['teacher_verdict'] == r['label'] for r in results) / n,
        'mobilevit_acc': sum(r['mobilevit_pred'] == r['label'] for r in results) / n,
    }
    with open(f'{args.out}/test_predictions.jsonl', 'w', encoding='utf-8') as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    with open(f'{args.out}/test_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(summary)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--model', default=MODEL_ID)
    ap.add_argument('--epochs', type=float, default=3)
    ap.add_argument('--lr', type=float, default=2e-4)
    ap.add_argument('--rank', type=int, default=16)
    ap.add_argument('--eval-only', action='store_true')
    ap.add_argument('--resume', nargs='?', const=True, default=None,
                    help='체크포인트에서 이어서 학습한다. 경로를 주거나, 빈 값이면 최신 것')
    ap.add_argument('--side', type=int, default=SIDE,
                    help='이미지 한 변(px). 392=장당 196토큰. 낮추면 VRAM이 준다')
    ap.add_argument('--workers', type=int, default=0 if os.name == 'nt' else 2,
                    help='DataLoader 워커. 윈도우는 spawn 비용 때문에 0이 낫다')
    ap.add_argument('--max-new-tokens', type=int, default=1800,
                    help='상세 설명은 1200~1600토큰이라 400이면 잘린다')
    args = ap.parse_args()
    # T4는 bf16을 지원하지 않는다
    args.dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    evaluate(args) if args.eval_only else train(args)

In [ ]:
%%writefile /content/repo/distill/vlm_server.py
"""증류된 Qwen2-VL 학생을 상주시켜 Streamlit 앱에 평가문을 스트리밍한다.

torch 환경(로컬 venv-train 또는 Colab)에서 돈다. 앱은 TensorFlow venv라 모델을 직접 못
올리므로 HTTP로 붙는다. 모델은 서버가 뜰 때 한 번만 올린다.

    venv-train\\Scripts\\python.exe distill\\vlm_server.py [--port 8502] [--quant auto|4bit|none]

GET  /health    {"ready": bool, "device": ..., "adapter": ..., "last": {...}}
POST /generate  {"images": {"orig": <b64>, "cam": <b64>, "crop": <b64>}, "meta": {...}}
                → 생성문을 글자가 나오는 대로 흘려보낸다 (text/plain, chunked)

이미지를 요청에 직접 싣기 때문에 서버가 다른 컴퓨터(Colab, AWS)에 있어도 된다.

정밀도(--quant auto): GPU 메모리가 12GB 이상이면 양자화 없이 fp16/bf16(작은 배치에선
4비트보다 빠르다), 그보다 작으면 학습 때와 같은 4비트, GPU가 없으면 CPU bf16.
QLoRA 어댑터는 양자화하지 않은 베이스에 붙여도 그대로 쓸 수 있다.
"""
import argparse
import base64
import io
import json
import os
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import torch
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from export_dataset import user_text
from train_qlora import MODEL_ID, build_messages

HERE = os.path.dirname(os.path.abspath(__file__))
# 5에포크판(final)이 없으면 3에포크판을 쓴다. test 정확도는 둘 다 78%로 같다.
ADAPTERS = [os.path.join(HERE, 'qwen2vl-distill', d) for d in ('final', 'final-3ep')]
# 학습 때 이미지 한 변. 다르면 학생이 못 보던 토큰 수가 들어간다.
SIDE = 336
# 학습 데이터와 같은 순서: 원본, 히트맵, 크롭
ORDER = ('orig', 'cam', 'crop')

state = {'ready': False, 'device': None, 'adapter': None, 'last': None}
lock = threading.Lock()  # 한 번에 한 장만 생성한다


def pick_adapter(explicit=None):
    for d in ([explicit] if explicit else ADAPTERS):
        f = os.path.join(d, 'adapter_model.safetensors')
        # LFS를 못 받으면 130바이트짜리 포인터 파일만 남는다
        if os.path.exists(f) and os.path.getsize(f) > 1_000_000:
            return d
    sys.exit(f'어댑터가 없다: {explicit or ADAPTERS}')


def load(args):
    from peft import PeftModel
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    if torch.cuda.is_available():
        dtype = {'bf16': torch.bfloat16, 'fp16': torch.float16}.get(
            args.dtype, torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
        vram = torch.cuda.get_device_properties(0).total_memory / 2**30
        quant = args.quant if args.quant != 'auto' else ('none' if vram >= 12 else '4bit')
        name = torch.cuda.get_device_name(0)
        if quant == '4bit':
            from transformers import BitsAndBytesConfig
            bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                     bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
            model = Qwen2VLForConditionalGeneration.from_pretrained(
                MODEL_ID, quantization_config=bnb, dtype=dtype, device_map='auto')
        else:
            model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, device_map='cuda')
        state['device'] = f'{name} {quant if quant == "4bit" else str(dtype).split(".")[-1]}'
    else:
        torch.set_num_threads(args.threads)
        model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
        state['device'] = f'cpu bf16 ({args.threads} threads)'
        quant = 'none'

    adapter = pick_adapter(args.adapter)
    model = PeftModel.from_pretrained(model, adapter)
    if quant != '4bit':
        # 어댑터를 가중치에 합친다. 안 합치면 토큰마다 LoRA 모듈 196개를 따로 거쳐 느리다.
        model = model.merge_and_unload()
    model = model.eval()
    px = SIDE * SIDE
    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=px, max_pixels=px)
    processor.tokenizer.padding_side = 'left'
    state['adapter'] = os.path.basename(adapter.rstrip('/\\'))
    return model, processor


def decode_images(images):
    return [Image.open(io.BytesIO(base64.b64decode(images[k]))).convert('RGB') for k in ORDER]


class Stop:
    """클라이언트가 끊으면 생성을 멈춘다 (Streamlit rerun 등)."""
    def __init__(self):
        self.flag = False

    def __call__(self, *a, **k):
        return self.flag


def make_handler(model, processor, max_new_tokens):
    from transformers import StoppingCriteriaList, TextIteratorStreamer

    class Handler(BaseHTTPRequestHandler):
        protocol_version = 'HTTP/1.1'  # chunked 전송은 1.1에서만 유효하다

        def log_message(self, fmt, *a):
            print(f'[vlm] {fmt % a}', file=sys.stderr, flush=True)

        def _json(self, code, obj):
            body = json.dumps(obj, ensure_ascii=False).encode('utf-8')
            self.send_response(code)
            self.send_header('Content-Type', 'application/json; charset=utf-8')
            self.send_header('Content-Length', str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            if self.path == '/health':
                self._json(200, state)
            else:
                self._json(404, {'error': 'not found'})

        def do_POST(self):
            if self.path != '/generate':
                return self._json(404, {'error': 'not found'})
            req = json.loads(self.rfile.read(int(self.headers['Content-Length'])))
            ex = {'images': list(ORDER), 'user_text': user_text(req['meta'])}
            prompt = processor.apply_chat_template(build_messages(ex, False), tokenize=False,
                                                   add_generation_prompt=True)
            enc = processor(text=[prompt], images=[decode_images(req['images'])],
                            return_tensors='pt').to(model.device)

            streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True,
                                            skip_special_tokens=True)
            stop = Stop()
            with lock:
                self.send_response(200)
                self.send_header('Content-Type', 'text/plain; charset=utf-8')
                self.send_header('Transfer-Encoding', 'chunked')
                # 터널(cloudflared)이 모아서 보내지 않게 한다
                self.send_header('Cache-Control', 'no-cache')
                self.send_header('X-Accel-Buffering', 'no')
                self.end_headers()
                t0, first, n = time.time(), None, 0
                t = threading.Thread(target=self._generate, args=(enc, streamer, stop))
                t.start()
                for piece in streamer:
                    if not piece or stop.flag:
                        continue
                    first = first or time.time() - t0
                    n += len(piece)
                    data = piece.encode('utf-8')
                    try:
                        self.wfile.write(f'{len(data):X}\r\n'.encode() + data + b'\r\n')
                        self.wfile.flush()
                    except OSError:
                        stop.flag = True  # 앱이 연결을 끊었다
                t.join()
                if not stop.flag:
                    self.wfile.write(b'0\r\n\r\n')
                total = time.time() - t0
                state['last'] = {'first_s': round(first or 0, 1), 'total_s': round(total, 1), 'chars': n}
                print(f'[vlm] 생성 완료: 첫 글자 {first or 0:.1f}초, 전체 {total:.1f}초, {n}자',
                      file=sys.stderr, flush=True)

        @torch.no_grad()
        def _generate(self, enc, streamer, stop):
            try:
                model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               streamer=streamer, stopping_criteria=StoppingCriteriaList([stop]))
            except Exception as e:  # 스트리머가 영원히 기다리지 않게 끝을 알린다
                print(f'[vlm] 생성 실패: {e!r}', file=sys.stderr, flush=True)
                streamer.text_queue.put(f'\n[생성 실패: {e}]')
                streamer.end()

    return Handler


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--port', type=int, default=8502)
    ap.add_argument('--host', default='127.0.0.1')
    ap.add_argument('--adapter', default=None, help='어댑터 폴더 (기본: final → final-3ep)')
    ap.add_argument('--quant', choices=['auto', '4bit', 'none'], default='auto')
    ap.add_argument('--dtype', choices=['auto', 'bf16', 'fp16'], default='auto',
                    help='GPU 계산 정밀도. T4는 bf16을 에뮬레이션하므로 fp16과 비교해 볼 것')
    ap.add_argument('--threads', type=int, default=int(os.environ.get('VLM_THREADS', 6)))
    ap.add_argument('--max-new-tokens', type=int, default=1800)
    args = ap.parse_args()

    print('[vlm] Qwen2-VL + LoRA 로드 중...', file=sys.stderr, flush=True)
    model, processor = load(args)
    state['ready'] = True
    print(f'[vlm] 준비 완료: {state["device"]}, 어댑터 {state["adapter"]}, '
          f'http://{args.host}:{args.port}', file=sys.stderr, flush=True)
    ThreadingHTTPServer((args.host, args.port),
                        make_handler(model, processor, args.max_new_tokens)).serve_forever()


if __name__ == '__main__':
    main()

## 4. LoRA 어댑터

5에포크 어댑터는 GitHub LFS에 없어서 **3에포크판(test 정확도 동일 78%)** 을 쓴다.

**이 셀을 돌리기 전에** 왼쪽 파일 패널(폴더 아이콘)의 업로드 버튼으로 PC의
`Image-Detection/distill/adapter-3ep.zip` 을 `/content` 에 올려 둔다. 없으면 파일 선택 창이 뜬다.
(조각 `adapter.partNN` 9개와 `adapter_config.json` 을 나눠 올려도 된다. 다 모이면 자동으로 합친다.)

In [ ]:
import glob, hashlib, os, shutil, zipfile
ADAPTER = '/content/repo/distill/qwen2vl-distill/final-3ep'
SHA = '1f4d48ef800abd4614150e1decf38fc709f3db7053dcd16cf908264767a89bf4'
os.makedirs(ADAPTER, exist_ok=True)
os.makedirs('/content/up', exist_ok=True)
have = lambda: all(os.path.exists(f'{ADAPTER}/{n}') for n in ('adapter_model.safetensors', 'adapter_config.json'))

def collect():
    # 파일 패널로 올린 것(/content)과 업로드 창으로 올린 것(/content/up)을 모두 본다
    found = lambda pat: sorted(glob.glob(f'/content/{pat}') + glob.glob(f'/content/up/{pat}'))
    for z in found('adapter*.zip'):
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/repo/distill/qwen2vl-distill')
    for c in found('adapter_config*.json')[:1]:
        shutil.copy(c, f'{ADAPTER}/adapter_config.json')
    parts = found('adapter.part??')
    if parts:
        print('조각', len(parts), '/ 9')
    if len(parts) == 9:
        with open(f'{ADAPTER}/adapter_model.safetensors', 'wb') as out:
            for p in parts:
                out.write(open(p, 'rb').read())

collect()
while not have():
    from google.colab import files
    os.chdir('/content/up')
    files.upload()
    collect()
os.chdir('/content/repo')
h = hashlib.sha256(open(f'{ADAPTER}/adapter_model.safetensors', 'rb').read()).hexdigest()
print('어댑터 해시', '일치' if h == SHA else f'불일치! {h}')

## 5. 서버 + 터널 실행
모델 로드에 1~2분 걸린다. 마지막 줄의 주소를 앱에 붙여 넣는다.

In [ ]:
import subprocess, time, re, requests
subprocess.run('pkill -f vlm_server.py; pkill -f cloudflared', shell=True)
time.sleep(2)
# T4 실측(2026-09-30): fp16 + 어댑터 병합으로 해설 한 건 약 58초. 병합 전에는 115초였다.
server = subprocess.Popen('exec python distill/vlm_server.py --host 127.0.0.1 --port 8502 --dtype fp16 '
                          '--adapter distill/qwen2vl-distill/final-3ep > /content/server.log 2>&1',
                          shell=True, cwd='/content/repo')
for _ in range(120):
    if server.poll() is not None:  # 서버가 죽었으면 기다리지 말고 로그를 보여 준다
        print(open('/content/server.log').read()[-3000:])
        raise RuntimeError('서버가 뜨지 못했다. 위 로그를 볼 것')
    try:
        if requests.get('http://127.0.0.1:8502/health', timeout=2).json().get('ready'):
            break
    except Exception:
        pass
    time.sleep(5)
print(requests.get('http://127.0.0.1:8502/health').json())

if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
open('/content/tunnel.log', 'w').close()  # 터널이 쓰기 전에 읽어도 죽지 않게 빈 파일을 먼저 만든다
tunnel = subprocess.Popen('/content/cloudflared tunnel --no-autoupdate --url http://127.0.0.1:8502 >> /content/tunnel.log 2>&1',
                          shell=True)
URL = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        URL = m.group(0); break
    time.sleep(1)
print('\n앱 사이드바에 붙여 넣을 주소:\n', URL)

## 6. 워밍업 + 속도 측정
회색 더미 이미지로 한 번 생성해 본다. 첫 요청은 CUDA 초기화 때문에 조금 더 느리다.

In [ ]:
import base64, io
from PIL import Image
def b64(color):
    buf = io.BytesIO(); Image.new('RGB', (384, 384), color).save(buf, 'JPEG'); return base64.b64encode(buf.getvalue()).decode()
req = {'images': {'orig': b64((128, 128, 128)), 'cam': b64((90, 90, 200)), 'crop': b64((140, 140, 140))},
       'meta': {'mobilevit_pred': 'AI', 'mobilevit_prob_real': 0.3, 'crop_box': [0, 0, 128, 128]}}
t0 = time.time(); first = None; text = ''
with requests.post(f'{URL}/generate', json=req, stream=True, timeout=600) as r:
    r.encoding = 'utf-8'
    for p in r.iter_content(chunk_size=None, decode_unicode=True):
        first = first or time.time() - t0; text += p
print(text[:300], '...')
print(f'\n터널 경유 — 첫 글자 {first:.1f}초, 전체 {time.time() - t0:.1f}초, {len(text)}자')
print('서버 측정:', requests.get(f'{URL}/health').json()['last'])

## 7. 로그 보기 (문제가 생겼을 때)

In [ ]:
!tail -20 /content/server.log
!tail -5 /content/tunnel.log